In [ ]:
%pip install psycopg2-binary pandas matplotlib python-dotenv

## 1. Carga de datos

In [ ]:
import psycopg2, pandas as pd, json, time, math
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv

try:
    from google.colab import userdata
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

VARIABLES = [
    "DB_NAME", "DB_USER", "DB_PASSWORD", "DB_HOST", "DB_PORT"
]

if EN_COLAB:
    config = {nombre: userdata.get(nombre) for nombre in VARIABLES}
else:
    load_dotenv(".env", override=False)
    config = {nombre: os.getenv(nombre) for nombre in VARIABLES}

faltantes = [nombre for nombre, valor in config.items() if not valor]
if faltantes:
    raise ValueError("Falta configurar: " + ", ".join(faltantes))


def connect_db():
    return psycopg2.connect(
        dbname=config["DB_NAME"],
        user=config["DB_USER"],
        password=config["DB_PASSWORD"],
        host=config["DB_HOST"],
        port=config["DB_PORT"],
        connect_timeout=10
    )


def fetch_data():
    conn = connect_db()
    df = pd.read_sql("SELECT id, contenido FROM noticias;", conn)
    conn.close()
    return df


noticias_df = fetch_data()

## 2. Preprocesamiento

### Tokenize

In [ ]:
import nltk
from ntlk.tokenize import word_tokenize
from nltk.corpus import stopwords
from ntlk.stem import PorterStemmer

nltk.download('punkt')
nltk.download('punkt_lab')
nltk.download('stopwords')

def preprocess(text):
    clean_text=text.lower()
    tokens=word_tokenize(clean_text)
    stop_words=set(stopwords.words('english'))
    filtered_text = [i for i in text if i not in stop_words]
    porter=PorterStemmer()

    stemmed_words=[porter.stem(filtered_text) for word in filtered_text]

    for original, stemmed in zip(filtered_text, stemmed_words):
        print(f"{original}->{stemmed}")
        